In [1]:
!pip install ucimlrepo

### Cargar el Dataset 'Bank Marketing'

In [2]:
from ucimlrepo import fetch_ucirepo
import pandas as pd

# Importar el dataset usando su ID
bank_marketing = fetch_ucirepo(id=222)

# Extraer los datos como DataFrames de pandas
X = bank_marketing.data.features
y = bank_marketing.data.targets

# Mostrar las primeras filas de las características (features)
display(X.head())

,age,job,marital,education,default,balance,housing,loan,contact,day_of_week,month,duration,campaign,pdays,previous,poutcome
0,58,management,married,tertiary,no,2143,yes,no,NaN,5,may,261,1,-1,0,NaN
1,44,technician,single,secondary,no,29,yes,no,NaN,5,may,151,1,-1,0,NaN
2,33,entrepreneur,married,secondary,no,2,yes,yes,NaN,5,may,76,1,-1,0,NaN
3,47,blue-collar,married,NaN,no,1506,yes,no,NaN,5,may,92,1,-1,0,NaN
4,33,NaN,single,NaN,no,1,no,no,NaN,5,may,198,1,-1,0,NaN


In [3]:
# Metadatos
print("Metadatos del dataset:")
print(bank_marketing.metadata)

# Información de las variables
print("\nInformación de las variables:")
display(bank_marketing.variables)

Metadatos del dataset:
{'uci_id': 222, 'name': 'Bank Marketing', 'repository_url': 'https://archive.ics.uci.edu/dataset/222/bank+marketing', 'data_url': 'https://archive.ics.uci.edu/static/public/222/data.csv', 'abstract': 'The data is related with direct marketing campaigns (phone calls) of a Portuguese banking institution. The classification goal is to predict if the client will subscribe a term deposit (variable y).', 'area': 'Business', 'tasks': ['Classification'], 'characteristics': ['Multivariate'], 'num_instances': 45211, 'num_features': 16, 'feature_types': ['Categorical', 'Integer'], 'demographics': ['Age', 'Occupation', 'Marital Status', 'Education Level'], 'target_col': ['y'], 'index_col': None, 'has_missing_values': 'yes', 'missing_values_symbol': 'NaN', 'year_of_dataset_creation': 2014, 'last_updated': 'Fri Aug 18 2023', 'dataset_doi': '10.24432/C5K306', 'creators': ['S. Moro', 'P. Rita', 'P. Cortez'], 'intro_paper': {'ID': 277, 'type': 'NATIVE', 'title': 'A data-driven ap

,name,role,type,demographic,description,units,missing_values
0,age,Feature,Integer,Age,None,None,no
1,job,Feature,Categorical,Occupation,"type of job (categorical: 'admin.','blue-colla...",None,no
2,marital,Feature,Categorical,Marital Status,"marital status (categorical: 'divorced','marri...",None,no
3,education,Feature,Categorical,Education Level,"(categorical: 'basic.4y','basic.6y','basic.9y'...",None,no
4,default,Feature,Binary,None,has credit in default?,None,no
5,balance,Feature,Integer,None,average yearly balance,euros,no
6,housing,Feature,Binary,None,has housing loan?,None,no
7,loan,Feature,Binary,None,has personal loan?,None,no
8,contact,Feature,Categorical,None,contact communication type (categorical: 'cell...,None,yes
9,day_of_week,Feature,Date,None,last contact day of the week,None,no


### 1. Instalar MLflow

Instalamos `mlflow` para poder registrar nuestros experimentos, parámetros, métricas y modelos.

In [4]:
!pip install mlflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.1/51.1 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 83.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.1/4.1 MB 91.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 79.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.4/228.4 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.5/136.5 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.7/148.7 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

### 2. Preprocesamiento de los Datos

El dataset tiene variables numéricas y categóricas. Crearemos un pipeline de preprocesamiento usando `ColumnTransformer` para estandarizar las numéricas y aplicar codificación One-Hot a las categóricas. También codificaremos la variable objetivo `y` a formato binario (0 y 1).

In [5]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Clonamos X e y para no alterar los DataFrames originales
X_prep = X.copy()
y_prep = y.copy()

# Codificar variable objetivo 'y' (yes -> 1, no -> 0)
y_prep['y'] = y_prep['y'].apply(lambda val: 1 if str(val).strip().lower() == 'yes' else 0)
y_numeric = y_prep['y'].values

# Identificar columnas por tipo
numeric_cols = X_prep.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = X_prep.select_dtypes(include=['object', 'category']).columns.tolist()

# Definir el transformador de columnas
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ]
)

# Dividir los datos en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(
    X_prep, y_numeric, test_size=0.2, random_state=42, stratify=y_numeric
)

### 3. Entrenamiento, Ajuste de Hiperparámetros (Grid Search) y Registro en MLflow

Definiremos un experimento en MLflow y ejecutaremos una búsqueda en rejilla (`GridSearchCV`) utilizando validación cruzada para:
1. **Regresión Logística**
2. **Random Forest Classifier**

Registraremos automáticamente los parámetros e hiperparámetros óptimos, así como las métricas clave (Accuracy, F1-Score y ROC-AUC) de cada modelo.

In [8]:
import mlflow
import mlflow.sklearn
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression

def run_mlflow_experiment(model_name, classifier, param_grid):
    with mlflow.start_run(run_name=model_name):
        # Crear el pipeline con preprocesamiento y el clasificador
        pipeline = Pipeline(steps=[
            ('preprocessor', preprocessor),
            ('classifier', classifier)
        ])

        # Ajustar los nombres de la rejilla de parámetros para que coincidan con el pipeline
        grid_params = {f'classifier__{key}': val for key, val in param_grid.items()}

        # Configurar la búsqueda en rejilla con validación cruzada (k=5)
        grid_search = GridSearchCV(
            pipeline,
            grid_params,
            cv=5,
            scoring='roc_auc',
            n_jobs=-1
        )

        grid_search.fit(X_train, y_train)

        # Evaluar el mejor modelo en el conjunto de prueba
        best_model = grid_search.best_estimator_
        y_pred = best_model.predict(X_test)

        from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
        accuracy = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        # Para ROC-AUC necesitamos probabilidades
        try:
            y_proba = best_model.predict_proba(X_test)[:, 1]
            roc_auc = roc_auc_score(y_test, y_proba)
        except AttributeError:
            roc_auc = roc_auc_score(y_test, y_pred)

        # Registrar parámetros y métricas en MLflow
        mlflow.log_params(grid_search.best_params_)
        mlflow.log_metric("cv_best_roc_auc_score", grid_search.best_score_)
        mlflow.log_metric("test_accuracy", accuracy)
        mlflow.log_metric("test_f1_score", f1)
        mlflow.log_metric("test_roc_auc", roc_auc)

        # Registrar el modelo entrenado usando pickle para evitar el problema de skops
        mlflow.sklearn.log_model(best_model, "model", serialization_format="pickle")

        print(f"=== {model_name} ===")
        print(f"Mejores parámetros: {grid_search.best_params_}")
        print(f"ROC-AUC en Test: {roc_auc:.4f}")
        print(f"Accuracy en Test: {accuracy:.4f}")
        print(f"F1-Score en Test: {f1:.4f}\n")

### 4. Ejecutar el pipeline de experimentos

Definimos el experimento en MLflow y ejecutamos el entrenamiento para ambos algoritmos con sus respectivas mallas de hiperparámetros.

In [9]:
from sklearn.ensemble import RandomForestClassifier

# Establecer el experimento activo en MLflow
mlflow.set_experiment("Bank_Marketing_Classification")

# Malla de parámetros para Regresión Logística
log_reg_params = {
    'C': [0.1, 1.0, 10.0],
    'max_iter': [1000]
}

# Malla de parámetros para Random Forest
rf_params = {
    'n_estimators': [50, 100],
    'max_depth': [10, 20]
}

# Ejecutar experimentos de entrenamiento, ajuste y registro en MLflow
run_mlflow_experiment("Logistic_Regression", LogisticRegression(), log_reg_params)
run_mlflow_experiment("Random_Forest", RandomForestClassifier(random_state=42), rf_params)

2026/10/19 05:18:29 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/10/19 05:18:29 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


=== Logistic_Regression ===
Mejores parámetros: {'classifier__C': 0.1, 'classifier__max_iter': 1000}
ROC-AUC en Test: 0.9061
Accuracy en Test: 0.9005
F1-Score en Test: 0.4438



2026/10/19 05:19:37 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/10/19 05:19:37 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


=== Random_Forest ===
Mejores parámetros: {'classifier__max_depth': 20, 'classifier__n_estimators': 100}
ROC-AUC en Test: 0.9281
Accuracy en Test: 0.9062
F1-Score en Test: 0.4873

